# Camada Silver Tratamento e Padronização

**Pipeline:** Análise da Alfabetização no Brasil
**Responsabilidade:** ler dados brutos da Bronze (Parquet), limpar, padronizar e normalizar chaves.

**Transformações aplicadas:**
- Remoção de duplicados
- Tratamento de valores nulos nas chaves
- Cast de tipos (integer, double, long)
- Padronização de siglas (upper + trim)
- Normalização de nomes de colunas (INEP → padrão do projeto)

**Princípio da camada:** a Silver não sabe a origem do dado (batch ou streaming) ela lê Parquet padronizado e entrega Parquet limpo para a Gold.

## 1. Configuração de Acesso

Conexão ao ADLS Gen2 via **Azure Key Vault** — a chave de acesso é lida em runtime e nunca fica exposta no código (boa prática de segurança).

In [ ]:
# ============================================================
# Configuração de acesso ao ADLS Gen2 via Azure Key Vault
# A chave é lida do Key Vault em runtime — nunca fica exposta no código
# ============================================================
from notebookutils import mssparkutils
import os

CONTA_STORAGE = os.getenv("STORAGE_ACCOUNT_NAME", "stalfalfabetizacao")
KEY_VAULT_URL = os.getenv("KEY_VAULT_URL", "https://kv-alfabetizacao.vault.azure.net/")
KEY_VAULT_SECRET_NAME = os.getenv("KEY_VAULT_SECRET_NAME", "storage-account-key")

# Busca a chave de acesso direto do Key Vault
CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    KEY_VAULT_URL,
    KEY_VAULT_SECRET_NAME
)

# Configura o Spark para usar a chave
spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

# Caminhos base de cada camada
CAMINHO_BRONZE = f"abfss://bronze@{CONTA_STORAGE}.dfs.core.windows.net/"
CAMINHO_SILVER = f"abfss://silver@{CONTA_STORAGE}.dfs.core.windows.net/"
CAMINHO_GOLD   = f"abfss://gold@{CONTA_STORAGE}.dfs.core.windows.net/"

print("Conexão configurada com sucesso!")
print(f"  Bronze : {CAMINHO_BRONZE}")
print(f"  Silver : {CAMINHO_SILVER}")
print(f"  Gold   : {CAMINHO_GOLD}")

StatementMeta(sparkpool, 5, 3, Finished, Available, Finished, False)

Conexão configurada com sucesso!
  Bronze : abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/
  Silver : abfss://silver@stalfalfabetizacao.dfs.core.windows.net/
  Gold   : abfss://gold@stalfalfabetizacao.dfs.core.windows.net/


### Leitura dos arquivos brutos da Bronze

## 2. Leitura da Camada Bronze

Leitura de todas as fontes em Parquet — batch (7 fontes) e streaming (4 fontes). A Silver parte sempre da Bronze limpa, nunca dos arquivos originais.

In [3]:
# ============================================================
# Leitura da camada Bronze (Parquet)
# A Silver não lê CSV nem JSON — só Parquet da Bronze
# Batch e streaming chegam aqui já padronizados
# ============================================================

# ── BATCH ────────────────────────────────────────────────────
df_uf             = spark.read.parquet(CAMINHO_BRONZE + "uf/")
df_municipio      = spark.read.parquet(CAMINHO_BRONZE + "municipio/")
df_meta_brasil    = spark.read.parquet(CAMINHO_BRONZE + "meta_brasil/")
df_meta_uf        = spark.read.parquet(CAMINHO_BRONZE + "meta_por_uf/")
df_meta_municipio = spark.read.parquet(CAMINHO_BRONZE + "meta_por_municipio/")
df_ts_municipio   = spark.read.parquet(CAMINHO_BRONZE + "ts_municipio/")
df_ts_estado      = spark.read.parquet(CAMINHO_BRONZE + "ts_estado/")

# ── STREAMING ────────────────────────────────────────────────
df_stream_aluno       = spark.read.parquet(CAMINHO_BRONZE + "parquet/stream_aluno/")
df_stream_municipio   = spark.read.parquet(CAMINHO_BRONZE + "parquet/stream_municipio/")
df_stream_indicadores = spark.read.parquet(CAMINHO_BRONZE + "parquet/stream_indicadores/")
df_stream_itens       = spark.read.parquet(CAMINHO_BRONZE + "parquet/stream_itens/")

print("Bronze lida com sucesso:")
print(f"  uf:                  {df_uf.count()} linhas")
print(f"  municipio:           {df_municipio.count()} linhas")
print(f"  meta_brasil:         {df_meta_brasil.count()} linhas")
print(f"  meta_uf:             {df_meta_uf.count()} linhas")
print(f"  meta_municipio:      {df_meta_municipio.count()} linhas")
print(f"  ts_municipio:        {df_ts_municipio.count()} linhas")
print(f"  ts_estado:           {df_ts_estado.count()} linhas")
print(f"  stream_aluno:        {df_stream_aluno.count()} eventos")
print(f"  stream_municipio:    {df_stream_municipio.count()} eventos")
print(f"  stream_indicadores:  {df_stream_indicadores.count()} eventos")
print(f"  stream_itens:        {df_stream_itens.count()} eventos")

StatementMeta(sparkpool, 5, 4, Finished, Available, Finished, False)

Bronze lida com sucesso:
  uf:                  145 linhas
  municipio:           23995 linhas
  meta_brasil:         3 linhas
  meta_uf:             54 linhas
  meta_municipio:      10704 linhas
  ts_municipio:        12416 linhas
  ts_estado:           80 linhas
  stream_aluno:        16 eventos
  stream_municipio:    11 eventos
  stream_indicadores:  11 eventos
  stream_itens:        16 eventos


## 3. Transformação e Limpeza

Aplicação das regras de qualidade em cada fonte: deduplicação, tratamento de nulos, cast de tipos e normalização de chaves. É o coração da camada Silver.

As fontes de **streaming** também são tratadas aqui — a Silver padroniza suas colunas (`NU_ANO` → `ano`, `CO_MUNICIPIO` → `id_municipio`, `SG_UF` → `sigla_uf`) para ficarem alinhadas com o batch.

In [4]:
# ============================================================
# SILVER — Limpeza e padronização de cada fonte
# Regras aplicadas:
#   - Remoção de duplicados
#   - Tratamento de nulos nas colunas chave
#   - Cast de tipos corretos
#   - Padronização de nomes de colunas
# ============================================================
from pyspark.sql.functions import col, trim, upper, when

# ── UF ──────────────────────────────────────────────────────
df_uf_silver = df_uf \
    .dropDuplicates() \
    .dropna(subset=["ano", "sigla_uf"]) \
    .withColumn("ano",              col("ano").cast("integer")) \
    .withColumn("sigla_uf",         upper(trim(col("sigla_uf")))) \
    .withColumn("serie",            col("serie").cast("integer")) \
    .withColumn("rede",             col("rede").cast("integer")) \
    .withColumn("taxa_alfabetizacao", col("taxa_alfabetizacao").cast("double")) \
    .withColumn("media_portugues",  col("media_portugues").cast("double"))

# ── MUNICÍPIO ────────────────────────────────────────────────
df_municipio_silver = df_municipio \
    .dropDuplicates() \
    .dropna(subset=["ano", "id_municipio"]) \
    .withColumn("ano",              col("ano").cast("integer")) \
    .withColumn("id_municipio",     col("id_municipio").cast("long")) \
    .withColumn("serie",            col("serie").cast("integer")) \
    .withColumn("rede",             col("rede").cast("integer")) \
    .withColumn("taxa_alfabetizacao", col("taxa_alfabetizacao").cast("double")) \
    .withColumn("media_portugues",  col("media_portugues").cast("double"))

# ── META BRASIL ──────────────────────────────────────────────
df_meta_brasil_silver = df_meta_brasil \
    .dropDuplicates() \
    .dropna(subset=["ano"]) \
    .withColumn("ano",  col("ano").cast("integer")) \
    .withColumn("rede", col("rede").cast("integer")) \
    .withColumn("taxa_alfabetizacao", col("taxa_alfabetizacao").cast("double"))

# ── META POR UF ──────────────────────────────────────────────
df_meta_uf_silver = df_meta_uf \
    .dropDuplicates() \
    .dropna(subset=["ano", "sigla_uf"]) \
    .withColumn("ano",      col("ano").cast("integer")) \
    .withColumn("sigla_uf", upper(trim(col("sigla_uf")))) \
    .withColumn("taxa_alfabetizacao", col("taxa_alfabetizacao").cast("double"))

# ── META POR MUNICÍPIO ───────────────────────────────────────
df_meta_municipio_silver = df_meta_municipio \
    .dropDuplicates() \
    .dropna(subset=["ano", "id_municipio"]) \
    .withColumn("ano",          col("ano").cast("integer")) \
    .withColumn("id_municipio", col("id_municipio").cast("long")) \
    .withColumn("taxa_alfabetizacao", col("taxa_alfabetizacao").cast("double"))

# ── TS_MUNICÍPIO (INEP) ──────────────────────────────────────
df_ts_municipio_silver = df_ts_municipio \
    .dropDuplicates() \
    .dropna(subset=["NU_ANO_AVALIACAO", "CO_MUNICIPIO"]) \
    .withColumn("NU_ANO_AVALIACAO",      col("NU_ANO_AVALIACAO").cast("integer")) \
    .withColumn("CO_MUNICIPIO",          col("CO_MUNICIPIO").cast("long")) \
    .withColumn("PC_ALUNO_ALFABETIZADO", col("PC_ALUNO_ALFABETIZADO").cast("double")) \
    .withColumnRenamed("NU_ANO_AVALIACAO", "ano") \
    .withColumnRenamed("CO_MUNICIPIO",     "id_municipio") \
    .withColumnRenamed("SG_UF",            "sigla_uf")

# ── TS_ESTADO (INEP) ─────────────────────────────────────────
df_ts_estado_silver = df_ts_estado \
    .dropDuplicates() \
    .dropna(subset=["NU_ANO_AVALIACAO", "CO_UF"]) \
    .withColumn("NU_ANO_AVALIACAO",      col("NU_ANO_AVALIACAO").cast("integer")) \
    .withColumn("PC_ALUNO_ALFABETIZADO", col("PC_ALUNO_ALFABETIZADO").cast("double")) \
    .withColumnRenamed("NU_ANO_AVALIACAO", "ano") \
    .withColumnRenamed("SG_UF",            "sigla_uf")

# Confirma contagens após limpeza
print("Contagens após limpeza Silver:")
print(f"  uf_silver:              {df_uf_silver.count()} linhas")
print(f"  municipio_silver:       {df_municipio_silver.count()} linhas")
print(f"  meta_brasil_silver:     {df_meta_brasil_silver.count()} linhas")
print(f"  meta_uf_silver:         {df_meta_uf_silver.count()} linhas")
print(f"  meta_municipio_silver:  {df_meta_municipio_silver.count()} linhas")
print(f"  ts_municipio_silver:    {df_ts_municipio_silver.count()} linhas")
print(f"  ts_estado_silver:       {df_ts_estado_silver.count()} linhas")

StatementMeta(sparkpool, 5, 5, Finished, Available, Finished, False)

Contagens após limpeza Silver:
  uf_silver:              145 linhas
  municipio_silver:       23995 linhas
  meta_brasil_silver:     3 linhas
  meta_uf_silver:         54 linhas
  meta_municipio_silver:  10704 linhas
  ts_municipio_silver:    12416 linhas
  ts_estado_silver:       80 linhas


## 4. Gravação da Camada Silver

Persistência em Parquet no container `silver/`. Formato colunar e comprimido — otimização de custo e performance (FinOps) para a leitura na Gold.

In [5]:
# ============================================================
# Gravação da camada Silver em formato Parquet
# Parquet = colunar + comprimido menor custo de storage e
# queries mais rápidas na Gold (FinOps na prática)
# mode("overwrite") permite reprocessar sem erro.
# ============================================================

df_uf_silver.write.mode("overwrite").parquet(
    CAMINHO_SILVER + "uf/"
)
df_municipio_silver.write.mode("overwrite").parquet(
    CAMINHO_SILVER + "municipio/"
)
df_meta_brasil_silver.write.mode("overwrite").parquet(
    CAMINHO_SILVER + "meta_brasil/"
)
df_meta_uf_silver.write.mode("overwrite").parquet(
    CAMINHO_SILVER + "meta_por_uf/"
)
df_meta_municipio_silver.write.mode("overwrite").parquet(
    CAMINHO_SILVER + "meta_por_municipio/"
)
df_ts_municipio_silver.write.mode("overwrite").parquet(
    CAMINHO_SILVER + "ts_municipio/"
)
df_ts_estado_silver.write.mode("overwrite").parquet(
    CAMINHO_SILVER + "ts_estado/"
)

print("Silver gravada com sucesso!")
print("Arquivos Parquet gerados em:")
print(f"  {CAMINHO_SILVER}")

StatementMeta(sparkpool, 5, 6, Finished, Available, Finished, False)

Silver gravada com sucesso!
Arquivos Parquet gerados em:
  abfss://silver@stalfalfabetizacao.dfs.core.windows.net/
